In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None, "display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
from src import config as C

# 09 — SQL Analizi

Veri analistliği/bilimciliği mülakatlarında SQL neredeyse her zaman sorulur. Temiz veriyi ve model skorlarını **SQLite** veritabanına aktarıp `sql/churn_analysis.sql` içindeki sorguları çalıştırıyoruz.

* `customers`: temiz veri
* `scored_customers`: temiz veri + dürüst risk skoru (eğitim satırları için out-of-fold, test satırları için hold-out)

Sorgular dosyada isimlendirilmiş; kullanılan SQL kavramları: `GROUP BY`, `HAVING`, `CASE WHEN`, `CTE (WITH)`, `RANK()`, `ROW_NUMBER()`, `NTILE()`, kümülatif `SUM() OVER`.

In [2]:
from src.sql_utils import build_connection, load_queries, run_all
con = build_connection()
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con).name.tolist())
queries = load_queries(); results = run_all(con)
def show(name, note=""):
    print(f"--- {name} ---"); print(queries[name]); display(results[name]); 
    if note: display(Markdown(note))

['customers', 'scored_customers']


## Genel özet ve sözleşme bazında churn + lift (CTE)

In [3]:
show('01_genel_ozet'); show('02_sozlesme_bazinda_churn_ve_lift', '**Lift**: segmentin churn oranı / genel churn oranı. Aylık sözleşme ≈ 1,4× riskli; iki yıllık ≈ 0,3×.')

--- 01_genel_ozet ---
SELECT COUNT(*)                          AS toplam_musteri,
       SUM(churn)                        AS ayrilan,
       ROUND(100.0 * AVG(churn), 2)      AS churn_orani_yuzde,
       ROUND(AVG(monthly_fee_tl), 2)     AS ort_aylik_ucret_tl
FROM customers;


,toplam_musteri,ayrilan,churn_orani_yuzde,ort_aylik_ucret_tl
0,1800,148,8.220,351.860


--- 02_sozlesme_bazinda_churn_ve_lift ---
-- CTE ile genel oranı bir kez hesaplayıp her segmentin genele göre kaç kat riskli olduğunu (lift) buluruz.
WITH genel AS (SELECT AVG(churn) AS oran FROM customers)
SELECT c.contract_type,
       COUNT(*)                                   AS musteri,
       SUM(c.churn)                               AS ayrilan,
       ROUND(100.0 * AVG(c.churn), 2)             AS churn_yuzde,
       ROUND(AVG(c.churn) / g.oran, 2)            AS lift
FROM customers c CROSS JOIN genel g
GROUP BY c.contract_type, g.oran
ORDER BY churn_yuzde DESC;


,contract_type,musteri,ayrilan,churn_yuzde,lift
0,Monthly,984,117,11.890,1.450
1,Annual,552,24,4.350,0.530
2,Two-Year,264,7,2.650,0.320


**Lift**: segmentin churn oranı / genel churn oranı. Aylık sözleşme ≈ 1,4× riskli; iki yıllık ≈ 0,3×.

## Plan × sözleşme matrisi (HAVING ile küçük grupları ele)

In [4]:
show('03_plan_sozlesme_matrisi')

--- 03_plan_sozlesme_matrisi ---
SELECT plan, contract_type,
       COUNT(*)                       AS musteri,
       ROUND(100.0 * AVG(churn), 2)   AS churn_yuzde
FROM customers
GROUP BY plan, contract_type
HAVING COUNT(*) >= 30          -- küçük gruplarda oranlar yanıltıcıdır
ORDER BY churn_yuzde DESC;


,plan,contract_type,musteri,churn_yuzde
0,Basic,Monthly,395,13.920
1,Premium,Monthly,224,12.500
2,Standard,Monthly,365,9.320
3,Basic,Annual,222,5.860
4,Premium,Two-Year,51,3.920
5,Premium,Annual,108,3.700
6,Basic,Two-Year,117,3.420
7,Standard,Annual,222,3.150
8,Standard,Two-Year,96,1.040


## Abonelik süresi dilimleri (CASE WHEN)

In [5]:
show('04_abonelik_suresi_dilimleri')

--- 04_abonelik_suresi_dilimleri ---
SELECT CASE WHEN tenure_months <= 6  THEN '1) 0-6 ay'
            WHEN tenure_months <= 12 THEN '2) 7-12 ay'
            WHEN tenure_months <= 24 THEN '3) 13-24 ay'
            WHEN tenure_months <= 48 THEN '4) 25-48 ay'
            ELSE '5) 49-72 ay' END     AS sure_dilimi,
       COUNT(*)                        AS musteri,
       SUM(churn)                      AS ayrilan,
       ROUND(100.0 * AVG(churn), 2)    AS churn_yuzde
FROM customers
GROUP BY sure_dilimi
ORDER BY sure_dilimi;


,sure_dilimi,musteri,ayrilan,churn_yuzde
0,1) 0-6 ay,130,20,15.380
1,2) 7-12 ay,153,20,13.070
2,3) 13-24 ay,284,34,11.970
3,4) 25-48 ay,607,48,7.910
4,5) 49-72 ay,626,26,4.150


## Ödeme gecikmesi — dikkat: ilişki monoton değil

In [6]:
show('05_odeme_gecikmesi', '3 gecikmede %18, 4 gecikmede %0 görünüyor ama 4 gecikmeli sadece **9 müşteri** var: küçük örneklemde oranlara güvenme.')

--- 05_odeme_gecikmesi ---
SELECT late_payments_6m,
       COUNT(*)                       AS musteri,
       ROUND(100.0 * AVG(churn), 2)   AS churn_yuzde
FROM customers
GROUP BY late_payments_6m
ORDER BY late_payments_6m;


,late_payments_6m,musteri,churn_yuzde
0,0,1044,7.570
1,1,565,9.200
2,2,160,8.130
3,3,22,18.180
4,4,9,0.000


3 gecikmede %18, 4 gecikmede %0 görünüyor ama 4 gecikmeli sadece **9 müşteri** var: küçük örneklemde oranlara güvenme.

## Bölge sıralaması (RANK pencere fonksiyonu)

In [7]:
show('06_bolge_siralamasi_pencere_fonksiyonu')

--- 06_bolge_siralamasi_pencere_fonksiyonu ---
-- RANK() pencere fonksiyonu: bölgeleri churn oranına göre sıralar (eksik bölge ayrı raporlanır).
SELECT RANK() OVER (ORDER BY AVG(churn) DESC)  AS sira,
       COALESCE(region, '(bilinmiyor)')         AS bolge,
       COUNT(*)                                 AS musteri,
       ROUND(100.0 * AVG(churn), 2)             AS churn_yuzde
FROM customers
GROUP BY region
ORDER BY sira;


,sira,bolge,musteri,churn_yuzde
0,1,(bilinmiyor),32,18.750
1,2,Central Anatolia,292,10.270
2,3,Black Sea,295,9.150
3,4,Mediterranean,287,9.060
4,5,Aegean,314,8.600
5,6,Eastern Anatolia,292,6.510
6,7,Marmara,288,4.510


## Memnuniyet × şikâyet

In [8]:
show('07_memnuniyet_dilimi_ve_sikayet')

--- 07_memnuniyet_dilimi_ve_sikayet ---
SELECT CASE WHEN satisfaction_score IS NULL THEN 'bilinmiyor'
            WHEN satisfaction_score < 4      THEN 'dusuk (<4)'
            WHEN satisfaction_score < 7      THEN 'orta (4-7)'
            ELSE 'yuksek (7+)' END           AS memnuniyet,
       CASE WHEN complaints_6m > 0 THEN 'sikayet var' ELSE 'sikayet yok' END AS sikayet,
       COUNT(*)                              AS musteri,
       ROUND(100.0 * AVG(churn), 2)          AS churn_yuzde
FROM customers
GROUP BY memnuniyet, sikayet
ORDER BY memnuniyet, sikayet;


,memnuniyet,sikayet,musteri,churn_yuzde
0,bilinmiyor,sikayet var,33,12.120
1,bilinmiyor,sikayet yok,49,4.080
2,dusuk (<4),sikayet var,105,21.900
3,dusuk (<4),sikayet yok,69,13.040
4,orta (4-7),sikayet var,343,9.910
5,orta (4-7),sikayet yok,622,8.200
6,yuksek (7+),sikayet var,129,4.650
7,yuksek (7+),sikayet yok,450,4.220


## Model skorlarıyla iş soruları

In [9]:
show('08_risk_kademesi_basarimi', 'Kademeler **tek yönlü** ayrışıyor: High kademesi müşterilerin ≈ %28\'i ama ayrılanların ≈ %60\'ı.'); show('09_riskteki_gelir')

--- 08_risk_kademesi_basarimi ---
-- Model skorları dürüst (out-of-fold / hold-out) olduğu için bu tablo gerçekçi bir başarı özetidir.
SELECT risk_tier,
       COUNT(*)                                          AS musteri,
       SUM(churn)                                        AS ayrilan,
       ROUND(100.0 * AVG(churn), 2)                      AS gercek_churn_yuzde,
       ROUND(100.0 * SUM(churn) / (SELECT SUM(churn) FROM scored_customers), 1) AS tum_ayrilanlarin_yuzdesi
FROM scored_customers
GROUP BY risk_tier
ORDER BY CASE risk_tier WHEN 'High' THEN 1 WHEN 'Medium' THEN 2 ELSE 3 END;


,risk_tier,musteri,ayrilan,gercek_churn_yuzde,tum_ayrilanlarin_yuzdesi
0,High,508,88,17.320,59.500
1,Medium,469,38,8.100,25.700
2,Low,823,22,2.670,14.900


Kademeler **tek yönlü** ayrışıyor: High kademesi müşterilerin ≈ %28'i ama ayrılanların ≈ %60'ı.

--- 09_riskteki_gelir ---
-- Aksiyon eşiğinin üstündeki (flagged) müşterilerin aylık gelir payı.
SELECT SUM(CASE WHEN flagged = 1 THEN monthly_fee_tl ELSE 0 END)                       AS riskli_aylik_gelir_tl,
       SUM(monthly_fee_tl)                                                              AS toplam_aylik_gelir_tl,
       ROUND(100.0 * SUM(CASE WHEN flagged = 1 THEN monthly_fee_tl ELSE 0 END) / SUM(monthly_fee_tl), 1) AS riskli_gelir_yuzde
FROM scored_customers;


,riskli_aylik_gelir_tl,toplam_aylik_gelir_tl,riskli_gelir_yuzde
0,"179,484.500","633,346.350",28.300


In [10]:
show('10_en_riskli_20_musteri')

--- 10_en_riskli_20_musteri ---
SELECT ROW_NUMBER() OVER (ORDER BY risk_score DESC) AS sira,
       customer_id, contract_type, plan, tenure_months, satisfaction_score,
       ROUND(risk_score, 3) AS risk_score, churn AS gercek_churn
FROM scored_customers
ORDER BY risk_score DESC
LIMIT 20;


,sira,customer_id,contract_type,plan,tenure_months,satisfaction_score,risk_score,gercek_churn
0,1,CUST101785,Monthly,Premium,12,2.100,0.615,0
1,2,CUST100778,Monthly,Premium,26,3.400,0.545,0
2,3,CUST101759,Monthly,Premium,1,2.600,0.540,0
3,4,CUST101538,Monthly,Standard,11,3.500,0.526,1
4,5,CUST101507,Monthly,Basic,1,1.800,0.493,0
5,6,CUST101505,Monthly,Basic,7,2.600,0.492,0
6,7,CUST101467,Monthly,Standard,7,4.000,0.483,0
7,8,CUST100302,Monthly,Basic,6,2.900,0.478,0
8,9,CUST100102,Monthly,Standard,10,3.200,0.470,1
9,10,CUST101751,Monthly,Premium,2,3.800,0.464,1


## Kümülatif yakalama (NTILE + pencere toplamı)

In [11]:
show('11_kumulatif_yakalama_pencere')

--- 11_kumulatif_yakalama_pencere ---
-- Müşterileri skora göre 10 eşit dilime böl (NTILE) ve ayrılanların kümülatif yakalanma oranını göster.
WITH dilim AS (
    SELECT NTILE(10) OVER (ORDER BY risk_score DESC) AS risk_dilimi, churn
    FROM scored_customers
), ozet AS (
    SELECT risk_dilimi, COUNT(*) AS musteri, SUM(churn) AS ayrilan
    FROM dilim GROUP BY risk_dilimi
)
SELECT risk_dilimi, musteri, ayrilan,
       ROUND(100.0 * SUM(ayrilan) OVER (ORDER BY risk_dilimi) / SUM(ayrilan) OVER (), 1) AS kumulatif_yakalanan_yuzde
FROM ozet
ORDER BY risk_dilimi;


,risk_dilimi,musteri,ayrilan,kumulatif_yakalanan_yuzde
0,1,180,47,31.800
1,2,180,22,46.600
2,3,180,25,63.500
3,4,180,12,71.600
4,5,180,14,81.100
5,6,180,10,87.800
6,7,180,7,92.600
7,8,180,3,94.600
8,9,180,5,98.000
9,10,180,3,100.000


In [12]:
con.close()